In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from tqdm import tqdm

In [2]:
from modules.pivot_data import PIVOT_DATA
feature, target = PIVOT_DATA()

    item_id  year  month  seq   hs4    weight  quantity     value
0  DEWLVASR  2022      1  1.0  3038   14858.0       0.0   32688.0
1  ELQGMQWE  2022      1  1.0  2002   62195.0       0.0  110617.0
2  AHMDUILJ  2022      1  1.0  2102   18426.0       0.0   72766.0
3  XIPPENFQ  2022      1  1.0  2501   20426.0       0.0   11172.0
4  FTSVTTSR  2022      1  1.0  2529  248000.0       0.0  143004.0


In [3]:
target.head()

item_id
AANGBULD      533478.0
AHMDUILJ      101317.0
ANWUJOKX           0.0
APQGTRMF       40608.0
ATLDMDBO    57090235.0
Name: 2025-07-01 00:00:00, dtype: float64

In [4]:
from modules.create_pairs import find_comovement_pairs
pairs = find_comovement_pairs(
    feature, 
    max_lag=12, 
    min_nonzero=6, 
    corr_threshold=0.4,
)
print(f"\n탐색된 공행성쌍 수: {len(pairs)}")

설정값:
- 상관계수 임계값: 0.4


공행성 쌍 탐색: 100it [00:10,  9.21it/s]


📊 검정 결과:
- 총 발견된 공행성 쌍: 2946

탐색된 공행성쌍 수: 2946


In [5]:
import pandas as pd
from modules.create_features import create_X_y
from modules.time_split  import create_time_series_datasets

def train_model_timesplit(datasets, pairs, scaler):
    """
    TimeSeriesSplit 데이터를 활용한 교차검증 예시
    마지막 fold의 스케일러를 저장하여 예측에 사용
    """
    models = []
    
    for i, dataset in enumerate(datasets):
        train_set = dataset['train_set']
        X_train_features, y_train = create_X_y(train_set, pairs)
        
        X_train_features = scaler.transform(X_train_features)
        
        model = LinearRegression()
        model.fit(X_train_features, y_train)
        models.append(model)
    
    return models

In [6]:

time_series_datasets = create_time_series_datasets(feature)

📅 시계열 데이터 기간:
  시작: 2022-01-01 00:00:00
  종료: 2025-06-01 00:00:00
  총 기간: 42개월
📊 교차검증 설정:
  총 시점 수: 42
  분할 수: 5
  최소 훈련 크기: 7
  Fold 1: 훈련(7개월) → 테스트(7개월)
  Fold 2: 훈련(14개월) → 테스트(7개월)
  Fold 3: 훈련(21개월) → 테스트(7개월)
  Fold 4: 훈련(28개월) → 테스트(7개월)
  Fold 5: 훈련(35개월) → 테스트(7개월)


In [7]:
from modules.scaler import get_scaler
# 마지막 fold의 학습 데이터로 스케일러 생성
X_train_features, y_train = create_X_y(time_series_datasets[-1]['train_set'], pairs)
scaler = get_scaler(fit_data=X_train_features)

In [8]:
models = train_model_timesplit(time_series_datasets, pairs, scaler)

In [9]:
def fold_predict(models, X_test):
    """모든 fold의 모든 모델 예측값을 평균"""
    all_predictions = []
    
    for model in models:
        pred = model.predict(X_test)
        all_predictions.append(pred)
    
    # 모든 예측값의 평균 (또는 median 사용 가능)
    result = np.median(all_predictions, axis=0)
    return result

In [10]:
from modules.create_features import calculate_features

def predict(pivot, pairs, models):
    months = pivot.columns.to_list()
    n_months = len(months)

    t_last = n_months - 1
    preds = []

    for row in pairs.itertuples(index=False):
        leader = row.선행품목
        follower = row.후행품목
        lag = int(row.최적지연기간)
        
        if leader not in pivot.index or follower not in pivot.index:
            continue

        a_series = pivot.loc[leader].values.astype(float)
        b_series = pivot.loc[follower].values.astype(float)

        (
            b_t,
            b_t_1,
            a_t_lag,
            # b_ma3,
            # b_std3,
            b_trend,
            # a_ma3,
            # a_std3,
            b_growth,
            a_growth,
        ) = calculate_features(a_series, b_series, t_last, lag)

        X_test = pd.DataFrame([[b_t, b_t_1, a_t_lag, float(row.최대상관계수), float(lag), row.상관계수_p값, b_trend, b_growth, a_growth]],columns=["현재총무역량", "직전달총무역량", "선행품목lag총무역량", "최대상관계수", "최적지연기간", "상관계수_p값", "후행품목추세", "후행품목성장률", "선행품목성장률"])
        X_test = scaler.transform(X_test)  # 예측 시에는 fit=False
        y_pred = fold_predict(models, X_test)[0]
        # 후처리
        y_pred = int(round(y_pred))

        preds.append({
            "leading_item_id": leader,
            "following_item_id": follower,
            "value": y_pred,
        })

    df_pred = pd.DataFrame(preds)
    return df_pred

In [11]:
submission = predict(feature,pairs,models)
    
print(f"\n✅ 예측 완료! 제출 데이터 크기: {len(submission)}개")
submission.head()


✅ 예측 완료! 제출 데이터 크기: 2946개


,leading_item_id,following_item_id,value
0,AANGBULD,APQGTRMF,580892
1,AANGBULD,BEZYMBBT,4114865
2,AANGBULD,DEWLVASR,806864
3,AANGBULD,DNMPSKTB,5739330
4,AANGBULD,ELQGMQWE,1160862


In [12]:
from score import print_validation_summary

print_validation_summary(target, submission, pairs)

🏆 대회 평가식 기반 모델 검증
🔍 Test 데이터 기반 Submission 검증
📊 기본 정보:
  • Test 데이터 (2025년 7월): 100개 품목
  • Test 데이터 중 거래량 > 0: 90개 품목
  • Submission 예측 쌍: 2946개
  • 활성 품목 (거래량 > 0): 90개
  • 검증 가능한 정답 쌍: 2748개

📋 대회 평가 결과:
  🏆 최종 점수: 0.7129
     ├─ F1-Score (60%): 0.9652
     └─ S2 (40%): 0.3343 (1-NMAE)

📈 상세 지표:
  • F1-Score: 0.9652
  • Precision: 0.9328
  • Recall: 1.0000
  • NMAE: 0.6657

🎯 매칭 분석:
  • True Positive (TP): 2748개 (정확히 매칭된 쌍)
  • False Positive (FP): 198개 (잘못 예측한 쌍)
  • False Negative (FN): 0개 (놓친 쌍)

📊 매칭된 쌍들의 예측 정확도:
  • 평균 상대오차: 0.6416
  • 중간값 상대오차: 0.7249
  • 최소 상대오차: 0.0014
  • 최대 상대오차: 1.0000
  • 10% 이내 정확도: 8.4%
  • 20% 이내 정확도: 18.7%

🏅 성능 등급: 🥈 양호 (0.6-0.8)

💡 개선 제안:
  🔸 NMAE 개선 방안:
    - 예측 모델 성능 향상 (XGBoost, RandomForest)
    - 더 많은 특성 엔지니어링
    - 이상치 제거 및 후처리 개선
  🔸 False Positive 감소:
    - 더 보수적인 공행성 기준 적용
    - 모델 confidence threshold 조정

🔥 최종 검증 결과 요약
🏆 대회 최종 점수: 0.7129
   ├─ 공행성 쌍 매칭 (F1): 0.9652 (가중치 60%)
   └─ 예측값 정확도 (S2): 0.3343 (가중치 40%)

📊 예상 성과: 🥉 상위 25% 예상

🔥 긴급

In [13]:
submission.to_csv('./submit.csv', index=False)